In [ ]:
API_KEY = ""
SAVE_FOLDER = "newsapiArticles"
START_DATE = "2025-11-01"
END_DATE = "2025-11-30"
TICKER = "AAPL"
LANGUAGE = "en"

In [2]:
from newspaper import Article, Config
from fake_useragent import UserAgent
import time


def scrapeArticle(url, retry_count=3, delay=2):
    for attempt in range(1, retry_count + 1):
        try:
            config = Config()
            ua = UserAgent()
            config.browser_user_agent = ua.random
            config.request_timeout = 10

            article = Article(url, language=LANGUAGE, config=config)
            article.download()
            article.parse()
            text = article.text

            if len(text.strip()) == 0:
                raise ValueError("Empty article text")

            return text

        except Exception as e:
            print(f"Attempt {attempt} failed for {url}: {e}")
            if attempt < retry_count:
                time.sleep(delay)
            else:
                print(f"Skipping article after {retry_count} failed attempts: {url}")
                return None


In [3]:
import os
import json

def processArticle(article, idx, page):
    url = article["url"]
    content = scrapeArticle(url)

    if not content:
        return None

    info = {
        "title": article.get("title"),
        "author": article.get("author"),
        "source": article.get("source", {}).get("name"),
        "publishedAt": article.get("publishedAt"),
        "url": url,
        "description": article.get("description"),
        "content": content,
    }

    date = article.get("publishedAt")[:10]
    filename = f"NewsAPI_{date}_{idx}.json"
    filepath = os.path.join(SAVE_FOLDER, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=4, ensure_ascii=False)

    return filename

In [4]:
import requests
import os
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

ticker_map = {
    "AAPL": ["AAPL", "AAPL.O", "AAPL.N", "NASDAQ:AAPL", "AAPL US", "AAPL UQ", "AAPL.OQ",
             "Apple Inc", "Apple Inc.", "Apple", "Apple Computers"],
}

terms = ticker_map.get(TICKER, [TICKER])
query = " OR ".join(f'"{t}"' for t in terms)

os.makedirs(SAVE_FOLDER, exist_ok=True)

start = datetime.strptime(START_DATE, "%Y-%m-%d")
end = datetime.strptime(END_DATE, "%Y-%m-%d")
delta = timedelta(days=1)

seen_urls = set()
saved_ctr = 0

current = start
while current <= end:
    print(f"Fetching articles from {current.strftime('%Y-%m-%d')}")
    params = {
        "apiKey": API_KEY,
        "q": query,
        "from": current.strftime("%Y-%m-%d"),
        "to": current.strftime("%Y-%m-%d"),
        "searchIn": "title,description",
        "sortBy": "publishedAt",
        "language": LANGUAGE,
        "pageSize": 100,
        "page": 1
    }

    response = requests.get("https://newsapi.org/v2/everything", params=params)
    data = response.json()

    if response.status_code != 200:
        print("Error:", data)
        break
    
    articles = data.get("articles", [])
    print(f"Found {len(articles)} articles")
    if not articles:
        current += timedelta(days=1)
        continue

    futures = {}
    with ThreadPoolExecutor(max_workers = 10) as executor:
        for idx, article in enumerate(articles):
            url = article["url"]
            if url in seen_urls:
                continue
            seen_urls.add(url)

            future = executor.submit(processArticle, article, idx, 1)
            futures[future] = url

        for future in as_completed(futures):
            result = future.result()
            if result:
                print("Saved:", result)
                saved_ctr += 1

    current += timedelta(days=1)

print(f"\nTotal articles saved: {saved_ctr}")


Fetching articles from 2023-10-01
Error: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2025-11-01, but you have requested 2023-10-01. You may need to upgrade to a paid plan.'}

Total articles saved: 0


In [8]:
link = ["https://www.thehindu.com/news/national/tamil-nadu/january-17-a-public-holiday-in-tamil-nadu/article17041135.ece1", "https://www.thehindu.com/news/national/what-do-indians-worry-about/article25056089.ece", "https://www.thehindu.com/news/cities/Hyderabad/five-year-old-brings-dying-mother-to-hospital/article22734506.ece"]

In [9]:
for url in link:
    content = scrapeArticle(url)
    if content:
        print(f"Content from {url}:\n{content[:500]}...\n")
    else:
        print(f"Failed to scrape {url}")

Attempt 1 failed for https://www.thehindu.com/news/national/tamil-nadu/january-17-a-public-holiday-in-tamil-nadu/article17041135.ece1: name 'LANGUAGE' is not defined
Attempt 2 failed for https://www.thehindu.com/news/national/tamil-nadu/january-17-a-public-holiday-in-tamil-nadu/article17041135.ece1: name 'LANGUAGE' is not defined
Attempt 3 failed for https://www.thehindu.com/news/national/tamil-nadu/january-17-a-public-holiday-in-tamil-nadu/article17041135.ece1: name 'LANGUAGE' is not defined
Skipping article after 3 failed attempts: https://www.thehindu.com/news/national/tamil-nadu/january-17-a-public-holiday-in-tamil-nadu/article17041135.ece1
Failed to scrape https://www.thehindu.com/news/national/tamil-nadu/january-17-a-public-holiday-in-tamil-nadu/article17041135.ece1
Attempt 1 failed for https://www.thehindu.com/news/national/what-do-indians-worry-about/article25056089.ece: name 'LANGUAGE' is not defined
Attempt 2 failed for https://www.thehindu.com/news/national/what-do-indians-w